In [ ]:
#---#| default_exp match.psm_match

# Match

Peak matching functionalities

In [ ]:
import numpy as np

from alphapeptdeep.peptdeep.match.psm_match import PepSpecMatch
from alpharaw import register_all_readers
from alpharaw.ms_data_base import ms_reader_provider

register_all_readers()

In [ ]:
#| hide
import io
import copy

from alphabase.psm_reader.pfind_reader import pFindReader
from alphabase.peptide.fragment import create_fragment_mz_dataframe_by_sort_precursor
from alpharaw.legacy_msdata.mgf import MGFReader

In [ ]:
#| hide
#unittest
mgf = io.StringIO("""
BEGIN IONS
TITLE=02445a_BA7-TUM_HLA_7_01_01-DDA-1h-R1.31809.31809.3.0.dta
CHARGE=3+
RTINSECONDS=0.5418930
PEPMASS=272.276336
103.92207 5457.3
104.20045 5051.4
108.70090 5891.7
113.94175 6442.6
116.92975 40506.3
116.93716 8945.5
128.37773 6427.8
131.95308 288352.6
133.93259 7344.6
138.44611 7326.1
139.00072 41556.8
140.00319 16738.8
140.99719 9493.8
145.93156 10209.3
145.94897 10497.8
147.94559 8206.3
147.96396 30552.8
148.95543 14654.7
149.96338 234207.8
150.95096 8306.0
157.01089 84638.9
158.01357 27925.7
159.00627 16084.7
163.94281 24751.1
163.95915 32203.3
165.95605 44458.0
165.97186 11530.2
166.99500 26432.2
167.97302 9216.7
181.95230 13858.8
191.95448 66152.7
192.95538 8408.9
193.07185 9092.8
193.95313 660574.9
194.95674 23452.8
194.99008 143940.9
200.00568 19510.8
200.99942 23678.7
204.30894 9406.1
209.96466 21853.6
211.96245 65351.0
218.90355 9149.6
223.91072 11300.2
238.89684 12108.8
243.93825 10150.2
243.97040 10987.7
244.94121 8744.2
246.90314 11556.3
271.93225 29430.0
271.99219 51184.4
272.19150 31960.4
272.98602 35844.1
273.94431 11031.8
284.47998 8191.3
290.00125 66212.4
290.99539 54064.7
293.89490 10005.0
407.06372 10838.2
464.36697 9715.4
633.40036 633.40036
698.81390 9711.7
707.301117 707.301117
END IONS
BEGIN IONS
TITLE=02445a_BA7-TUM_HLA_7_01_01-DDA-1h-R1.23862.23862.2.0.dta
CHARGE=2+
RTINSECONDS=0.6455220
PEPMASS=287.427959
103.34669 5304.0
104.66884 5639.7
113.42419 6258.3
118.84039 5837.5
119.93203 13977.3
130.69589 6876.2
133.94824 43094.3
134.30524 7671.5
135.96359 9031.3
138.99994 8329.7
146.95573 31143.9
147.96323 12176.5
150.95151 65859.3
151.95818 24384.2
157.01105 19241.5
157.34985 7532.5
161.08838 7843.9
161.94234 20119.7
162.95146 60110.4
163.95877 183305.5
164.96657 13647.5
174.95139 150331.9
175.95258 21393.4
178.94460 11433.1
179.95316 13650.5
180.96204 15353.5
190.94572 30418.9
191.95422 61914.1
192.61461 8642.1
192.94395 12331.4
192.96207 132342.5
193.96318 19303.0
209.04164 25149.6
209.96368 154185.0
209.98361 12353.5
213.86244 11541.3
224.93071 12903.0
228.92879 8773.6
241.86043 135357.5
242.86113 20805.2
242.94327 26679.4
243.95219 29569.9
244.92361 12153.5
246.90300 16650.3
252.96521 73484.3
253.96646 11527.5
286.85858 10166.4
287.94186 18763.2
303.87665 39189.3
304.88116 11976.0
321.89087 97122.5
322.88867 28020.8
370.28696 9008.2
389.82578 13277.0
407.83545 12220.4
425.84872 13236.5
482.54852 10940.2
END IONS
BEGIN IONS
TITLE=02445a_BA7-TUM_HLA_7_01_01-DDA-1h-R1.23431.23431.2.0.dta
CHARGE=2+
RTINSECONDS=0.6455220
PEPMASS=287.427959
103.34669 5304.0
104.66884 5639.7
END IONS
BEGIN IONS
TITLE=02445a_BA7-TUM_HLA_7_01_01-DDA-1h-R1.32733.32733.2.0.dta
CHARGE=2+
RTINSECONDS=0.6455220
PEPMASS=287.427959
103.34669 5304.0
104.66884 5639.7
402.705571 402.705571
END IONS
BEGIN IONS
TITLE=02445a_BA7-TUM_HLA_7_01_01-DDA-1h-R1.23669.23669.2.0.dta
CHARGE=2+
RTINSECONDS=0.6455220
PEPMASS=287.427959
END IONS
""")

ms_file_dict = {
    'raw': copy.deepcopy(mgf),
    'raw1': copy.deepcopy(mgf),
}

psmlabel_str = '''File_Name	Sequence	Modification	Charge	Scan_No	Proteins	Q-value	Target/Decoy	Final_Score
raw.31809.31809.2.0.dta	PSTDLLMLK	2,Phospho[S];7,Oxidation[M];	2	31809	Prot	0	target	100
raw.23862.23862.2.0.dta	HTAYSDFLSDK		2	23862	Prot	0	target	100
raw.23431.23431.2.0.dta	HTAYSDFLSDK		2	23431	Prot	0	target	100
raw.32733.32733.2.0.dta	HFALFSTDVTK		2	32733	Prot	0	target	100
raw.23669.23669.2.0.dta	HTAYSDFLSDK		2	23669	Prot	0	target	100
raw1.31809.31809.2.0.dta	PSTDLLMLK	2,Phospho[S];7,Oxidation[M];	2	31809	Prot	0	target	100
raw1.23862.23862.2.0.dta	HTAYSDFLSDK		2	23862	Prot	0	target	100
raw1.23431.23431.2.0.dta	HTAYSDFLSDK		2	23431	Prot	0	target	100
raw1.32733.32733.2.0.dta	HFALFSTDVTK		2	32733	Prot	0	target	100
'''
reader = pFindReader()
reader.import_file(io.StringIO(psmlabel_str))
psm_df = reader.psm_df
matching = PepSpecMatch()
matching.match_ms2_multi_raw(psm_df, ms_file_dict, 'mgf')
merrs = matching.matched_mz_err_df.values
#np.sum(matching.matched_intensity_df.values!=0,axis=1)
assert len(merrs[~np.isinf(merrs)])==6
assert np.count_nonzero(matching.matched_intensity_df.values)==6

100%|██████████| 2/2 [00:01<00:00,  1.41it/s]


In [ ]:
#| hide
mgf_reader = ms_reader_provider.get_reader('mgf')
mgf_reader.import_raw(copy.deepcopy(mgf))
ms_file_dict = {'raw': mgf_reader}
mgf_reader1 = ms_reader_provider.get_reader('mgf')
mgf_reader1.import_raw(copy.deepcopy(mgf))
ms_file_dict['raw1'] = mgf_reader1
matching.match_ms2_multi_raw(psm_df, ms_file_dict, 'mgf')
merrs = matching.matched_mz_err_df.values
assert np.count_nonzero(matching.matched_intensity_df.values) == 6
assert len(merrs[~np.isinf(merrs)]) == 6

100%|██████████| 2/2 [00:00<00:00, 174.90it/s]


In [ ]:
#| hide
ms_file_dict = {
    'raw': copy.deepcopy(mgf),
}
reader = pFindReader()
reader.import_file(io.StringIO(psmlabel_str))
psm_df = reader.psm_df
psm_df = psm_df[~psm_df.raw_name.str.startswith('raw1')].copy()
matching = PepSpecMatch()
matching.match_ms2_multi_raw(psm_df, ms_file_dict, 'mgf')
matching.load_ms_data(copy.deepcopy(mgf), 'mgf')
df, frag_mz_df, frag_inten_df, frag_merr_df = matching.match_ms2_one_raw(
    psm_df
)
assert (matching.fragment_mz_df==frag_mz_df).values.all()
assert (matching.matched_intensity_df==frag_inten_df).values.all()
assert (matching.matched_mz_err_df==frag_merr_df).values.all()

100%|██████████| 1/1 [00:00<00:00, 111.73it/s]
